<a href="https://colab.research.google.com/github/Andru-1987/data_science_ii_96085/blob/main/01_semana/clase_practica_01_semana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conexiones bases de datos
> Iniciando conexion con bases de datos relaciones: `sqlite`
---

In [6]:
# una forma de asegurarnos que tenemos esa dependencia
! pip freeze | grep 'sqlite'

aiosqlite==0.22.1


In [24]:
!pip install sqlite-web

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.4/781.4 kB 11.8 MB/s eta 0:00:00


In [7]:
import sqlite3

In [48]:
DB_PATH_NAME:str = "blog.database.sqlite"
DB_PATH_NAME_SHORT:str = "blog_short.database.sqlite"


In [22]:
DDL_BASE_BLOG:str =  '''
CREATE TABLE IF NOT EXISTS user(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER,
    nationality TEXT
);

CREATE TABLE IF NOT EXISTS post(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title VARCHAR(25) NOT NULL,
    description TEXT NOT NULL,
    user_id INTEGER,
    FOREIGN KEY (user_id) REFERENCES user(id)
);


CREATE TABLE IF NOT EXISTS comment(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    comment_text TEXT NOT NULL,
    user_id INTEGER,
    post_id INTEGER,
    FOREIGN KEY (user_id) REFERENCES user(id),
    FOREIGN KEY (post_id) REFERENCES post(id)
);

CREATE TABLE IF NOT EXISTS like(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER,
    post_id INTEGER,
    FOREIGN KEY (user_id) REFERENCES user(id),
    FOREIGN KEY (post_id) REFERENCES post(id)
);
'''

ddl_queries_list = DDL_BASE_BLOG.split(";")




DATABASE_SEEDER:str = '''
INSERT INTO user (name, age, nationality) VALUES
('Juan Pérez', 28, 'Argentina'),
('María Gómez', 32, 'Chile'),
('Carlos López', 24, 'Uruguay'),
('Ana Torres', 29, 'Perú'),
('Lucía Fernández', 35, 'Argentina'),
('Pedro Martínez', 41, 'México'),
('Sofía Herrera', 22, 'Colombia'),
('Diego Castro', 30, 'España'),
('Valentina Ruiz', 27, 'Ecuador'),
('Martín Silva', 38, 'Uruguay');


INSERT INTO post (title, description, user_id) VALUES
('Mi primer post', 'Este es mi primer post en la plataforma.', 1),
('Viaje al sur', 'Comparto algunas fotos de mi viaje.', 2),
('Python Tips', 'Algunos consejos útiles para programar.', 3),
('Receta Pizza', 'Mi receta favorita de pizza casera.', 4),
('Entrenamiento', 'Cómo organizo mi rutina semanal.', 5),
('SQLite Básico', 'Introducción a SQLite para principiantes.', 6),
('Docker Fácil', 'Primeros pasos utilizando Docker.', 7),
('Machine Learning', 'Modelos supervisados explicados.', 8),
('Backend Node', 'Express y buenas prácticas.', 9),
('Vacaciones', 'Resumen de mis vacaciones de invierno.', 10);

INSERT INTO comment (comment_text, user_id, post_id) VALUES
('Excelente publicación.', 2, 1),
('Muy interesante.', 3, 1),
('Gracias por compartir.', 4, 2),
('Lo voy a probar.', 5, 3),
('Muy buenos consejos.', 6, 3),
('Me encantó la receta.', 7, 4),
('Excelente explicación.', 8, 6),
('Aprendí bastante.', 9, 8),
('Buen contenido.', 10, 9),
('Espero la segunda parte.', 1, 7);

INSERT INTO "like" (user_id, post_id) VALUES
(2, 1),
(3, 1),
(4, 2),
(5, 2),
(6, 3),
(7, 4),
(8, 5),
(9, 6),
(10, 7),
(1, 8);
'''

blog_insert_queries:list[str] = DATABASE_SEEDER.split(";")



In [8]:
class SqliteConnection:
    def __init__(self, db_name: str):
        self.db_name = db_name
        self.connection = None

    def __enter__(self):
        self.connection = sqlite3.connect(self.db_name)
        return self.connection

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            self.connection.commit()
        else:
            self.connection.rollback()

        self.connection.close()

        return False

In [21]:
with SqliteConnection(DB_PATH_NAME) as conn:
    # puntero que va a estar escuchando alguna accion del lado del cliente
    cursor = conn.cursor()
    for query in ddl_queries_list:
        cursor.execute(query)
    cursor.execute("SELECT name FROM sqlite_schema WHERE type='table' AND name NOT LIKE 'sqlite_%';")
    tables = cursor.fetchall()
    print(tables)

[('user',), ('post',), ('comment',), ('like',)]


In [23]:
with SqliteConnection(DB_PATH_NAME) as conn:
    cursor = conn.cursor()
    for query in blog_insert_queries:
        cursor.execute(query)

# Consultar a la base de datos

In [35]:
import pandas as pd

with SqliteConnection(DB_PATH_NAME) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM user")
    users = cursor.fetchall()

    cursor.execute("SELECT name FROM pragma_table_info('user')")
    table_columns_name = [ column[0] for column in cursor.fetchall()]
    print(table_columns_name)
    ## convertir en un dataframe con pandas
    df = pd.DataFrame(users, columns=table_columns_name)


df

['id', 'name', 'age', 'nationality']


,id,name,age,nationality
0,1,Juan Pérez,28,Argentina
1,2,María Gómez,32,Chile
2,3,Carlos López,24,Uruguay
3,4,Ana Torres,29,Perú
4,5,Lucía Fernández,35,Argentina
5,6,Pedro Martínez,41,México
6,7,Sofía Herrera,22,Colombia
7,8,Diego Castro,30,España
8,9,Valentina Ruiz,27,Ecuador
9,10,Martín Silva,38,Uruguay


In [46]:
QUERY_POST_WITH_USERS:str = '''
SELECT user.id AS user_id, user.name, post.id AS post_id, post.title, post.description
FROM post
JOIN user ON post.user_id = user.id
'''
QUERY_POST_COMMENTS_BY_USERS:str = '''
SELECT user.id AS user_id, user.name, comment.id AS comment_id, comment.comment_text
FROM post
JOIN user ON post.user_id = user.id
JOIN comment ON post.id = comment.post_id
'''


In [47]:
with SqliteConnection(DB_PATH_NAME) as conn:
    df = pd.read_sql_query(QUERY_POST_COMMENTS_BY_USERS, conn)

df

,user_id,name,comment_id,comment_text
0,1,Juan Pérez,1,Excelente publicación.
1,1,Juan Pérez,2,Muy interesante.
2,2,María Gómez,3,Gracias por compartir.
3,3,Carlos López,4,Lo voy a probar.
4,3,Carlos López,5,Muy buenos consejos.
5,4,Ana Torres,6,Me encantó la receta.
6,6,Pedro Martínez,7,Excelente explicación.
7,8,Diego Castro,8,Aprendí bastante.
8,9,Valentina Ruiz,9,Buen contenido.
9,7,Sofía Herrera,10,Espero la segunda parte.


In [42]:
with SqliteConnection(DB_PATH_NAME) as conn:
    cursor = conn.cursor()
    cursor.execute(QUERY_POST_WITH_USERS)
    posts = cursor.fetchall()

    columns = [ desc[0] for desc in cursor.description]
    ## convertir en un dataframe con pandas
    df = pd.DataFrame(posts, columns=columns)

df

,user_id,name,post_id,title,description
0,1,Juan Pérez,1,Mi primer post,Este es mi primer post en la plataforma.
1,2,María Gómez,2,Viaje al sur,Comparto algunas fotos de mi viaje.
2,3,Carlos López,3,Python Tips,Algunos consejos útiles para programar.
3,4,Ana Torres,4,Receta Pizza,Mi receta favorita de pizza casera.
4,5,Lucía Fernández,5,Entrenamiento,Cómo organizo mi rutina semanal.
5,6,Pedro Martínez,6,SQLite Básico,Introducción a SQLite para principiantes.
6,7,Sofía Herrera,7,Docker Fácil,Primeros pasos utilizando Docker.
7,8,Diego Castro,8,Machine Learning,Modelos supervisados explicados.
8,9,Valentina Ruiz,9,Backend Node,Express y buenas prácticas.
9,10,Martín Silva,10,Vacaciones,Resumen de mis vacaciones de invierno.


In [49]:
with SqliteConnection(DB_PATH_NAME_SHORT) as conn:
    cursor = conn.cursor()
    cursor.executescript(DDL_BASE_BLOG)
    conn.commit()


## NO SQL

Rest api : json -> [ schema | tabular | procesar ]-> dataframe (tabla)

In [88]:
import json
from tinydb import TinyDB, Query

In [51]:
! pip install tinydb

In [58]:
from textwrap import indent
class TinyDBConnection:
    def __init__(self, db_name: str):
        self.db_name = db_name
        self.db = None

    def __enter__(self):
        self.db = TinyDB(self.db_name, indent=4)
        return self.db

    def __exit__(self, exc_type, exc_value, traceback):
        self.db.close()
        return False

In [60]:
with TinyDBConnection("blog.database.json") as db:
    users = db.table("user")
    posts = db.table("post")


    users.insert_multiple(
        [
            {"name": "James", "age": 25, "gender": "hombre", "nationality": "USA"},
            {"name": "Leila", "age": 32, "gender": "mujer", "nationality": "France"},
            {"name": "Brigitte", "age": 35, "gender": "mujer", "nationality": "England"},
            {"name": "Mike", "age": 40, "gender": "hombre", "nationality": "Denmark"},
            {"name": "Elizabeth", "age": 21, "gender": "mujer", "nationality": "Canada"},
        ]
    )

    posts.insert_multiple(
        [
            {
                "title": "Feliz",
                "description": "Me siento feliz hoy",
                "user_id": 1,
                "comments": [{"text": "Cuenta conmigo", "user_id": 1}],
                "likes": [1, 2],
            },
            {
                "title": "Caliente",
                "description": "El clima esta caliente hoy",
                "user_id": 2,
                "comments": [],
                "likes": [4],
            },
            {
                "title": "Ayuda",
                "description": "Necesito ayuda en esto",
                "user_id": 2,
                "comments": [
                    {"text": "Que tipo de ayuda?", "user_id": 5},
                    {"text": "Te ayudo con tu tesis?", "user_id": 2},
                ],
                "likes": [2],
            },
            {
                "title": "Buenas noticias",
                "description": "Me casare pronto",
                "user_id": 1,
                "comments": [
                    {"text": "Felicitaciones", "user_id": 2},
                    {"text": "Muchas felicitaciones", "user_id": 5},
                ],
                "likes": [5, 2],
            },
            {
                "title": "Juego interesante",
                "description": "Fue genial jugar al tenis",
                "user_id": 5,
                "comments": [{"text": "Estuve jugando con Rafael", "user_id": 4}],
                "likes": [1],
            },
            {
                "title": "Fiesta",
                "description": "Alguno quiere venir a esta fiesta hoy?",
                "user_id": 3,
                "comments": [],
                "likes": [1, 3],
            },
        ]
    )



In [61]:
print("Posts con mas de 1 like")


Posts con mas de 1 like


In [62]:
post_query = Query()

In [87]:
with TinyDBConnection("blog.database.json") as db:
    data = db.table('post').search(
        post_query.likes.test(lambda likes: len(likes) == 1 )
    )

    posts_df_normalize = pd.json_normalize(data, record_path=["comments"], meta=["title","description","user_id"], meta_prefix = 'posts_')


## directamente transformado en un dict de python
data[0].get("likes")
posts_df_normalize

,text,user_id,posts_title,posts_description,posts_user_id
0,Que tipo de ayuda?,5,Ayuda,Necesito ayuda en esto,2
1,Te ayudo con tu tesis?,2,Ayuda,Necesito ayuda en esto,2
2,Estuve jugando con Rafael,4,Juego interesante,Fue genial jugar al tenis,5
3,Que tipo de ayuda?,5,Ayuda,Necesito ayuda en esto,2
4,Te ayudo con tu tesis?,2,Ayuda,Necesito ayuda en esto,2
5,Estuve jugando con Rafael,4,Juego interesante,Fue genial jugar al tenis,5
